# Smith-Waterman Local Alignment Implementation

### Imports

In [1]:
# Import
import numpy as np

### Scoring function

In [2]:
def cal_score(matrix, seq1, seq2, i, j, match, mismatch, gap):
    '''Calculate score for position (i,j) in scoring matrix, also record move to trace back
    '''
    # Calculate diagonal score
    if seq1[i - 1] == seq2[j - 1]:
        diag_score = matrix[i - 1, j - 1] + match
    else:
        diag_score = matrix[i - 1, j - 1] + mismatch

    # Calculate gap scores
    up_score = matrix[i - 1, j] + gap
    left_score = matrix[i, j - 1] + gap

    # Determine the maximum score
    score = max(0, diag_score, up_score, left_score)

    # Record the highest score for traceback
    if score == 0:
        move = 0 # End of local alignment
    elif score == diag_score:
        move = 1 # Diagonal for match/mismatch
    elif score == up_score:
        move = 2 # Up for gap in seq2
    else:
        move = 3 # Left for gap in seq1

    return score, move

print("Successfully defined cal_score function.")

Successfully defined cal_score function.


### Traceback function

In [3]:
def traceback(seq1, seq2, traceback_matrix, scoring_matrix, maximum_position):
    '''Find the optimal path through the scoring matrix starting from the highest score
    '''
    aligned_seq1 = []
    aligned_seq2 = []
    curr_i, curr_j = maximum_position

    # Trace back until we reach a score of zero
    while curr_i > 0 and curr_j > 0 and scoring_matrix[curr_i, curr_j] > 0:
        move = traceback_matrix[curr_i, curr_j]

        if move == 1: # Diagonal for match or mismatch
            aligned_seq1.append(seq1[curr_i - 1])
            aligned_seq2.append(seq2[curr_j - 1])
            curr_i -= 1
            curr_j -= 1
        elif move == 2: # Up for gap in sequence 2
            aligned_seq1.append(seq1[curr_i - 1])
            aligned_seq2.append("-")
            curr_i -= 1
        elif move == 3: # Left for gap in sequence 1
            aligned_seq1.append("-")
            aligned_seq2.append(seq2[curr_j - 1])
            curr_j -= 1
        else:
            break

    # Traceback builds them from end to start
    return "".join(reversed(aligned_seq1)), "".join(reversed(aligned_seq2))

print("Successfully defined traceback function.")

Successfully defined traceback function.


### Main Smith Waterman Implementation

In [4]:
def smith_waterman(seq1, seq2, match=1, mismatch=-1, gap=-1):
    '''Implements Smith-Waterman algorithm for local sequence alignment
    '''
    m, n = len(seq1), len(seq2)

    # Initialize matrices with zeros
    score_matrix = np.zeros((m + 1, n + 1), dtype=int)
    traceback_matrix = np.zeros((m + 1, n + 1), dtype=int)

    max_score = 0
    max_pos = (0, 0)

    # Fill the scoring matrix using dynamic programming
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            score, move = cal_score(score_matrix, seq1, seq2, i, j, match, mismatch, gap)
            score_matrix[i, j] = score
            traceback_matrix[i, j] = move

            # Update the highest score found in the matrix
            if score >= max_score:
                max_score = score
                max_pos = (i, j)

    # Reconstruct the alignment starting from the peak score
    aligned1, aligned2 = traceback(seq1, seq2, traceback_matrix, score_matrix, max_pos)

    return aligned1, aligned2, score_matrix

print("Successfully defined smith_waterman main function.")

Successfully defined smith_waterman main function.


### Test

In [5]:
# Test Case
seq1 = 'TACTTAG'
seq2 = 'CACATTAA'

# Execute the algorithm
aligned_seq1, aligned_seq2, score_matrix = smith_waterman(seq1, seq2)

# Find the maximum score in the matrix
alignment_score = np.max(score_matrix)

print("--- Smith-Waterman Local Alignment Results ---")
print(f"Sequence 1 Alignment: {aligned_seq1}")
print(f"Sequence 2 Alignment: {aligned_seq2}")
print(f"Final alignment score: {alignment_score}")
print("\nFinal Scoring Matrix:")
print(score_matrix)

--- Smith-Waterman Local Alignment Results ---
Sequence 1 Alignment: AC-TTA
Sequence 2 Alignment: ACATTA
Final alignment score: 4

Final Scoring Matrix:
[[0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 0 0]
 [0 0 1 0 1 0 0 2 1]
 [0 1 0 2 1 0 0 1 1]
 [0 0 0 1 1 2 1 0 0]
 [0 0 0 0 0 2 3 2 1]
 [0 0 1 0 1 1 2 4 3]
 [0 0 0 0 0 0 1 3 3]]
